In [1]:
import os
for root, dirs, files in os.walk('/kaggle/input/models'):
    print(root, dirs, files)

/kaggle/input/models ['metaresearch'] []
/kaggle/input/models/metaresearch ['dinov2'] []
/kaggle/input/models/metaresearch/dinov2 ['pytorch'] []
/kaggle/input/models/metaresearch/dinov2/pytorch ['small'] []
/kaggle/input/models/metaresearch/dinov2/pytorch/small ['1'] []
/kaggle/input/models/metaresearch/dinov2/pytorch/small/1 [] ['config.json', 'preprocessor_config.json', 'README.md', 'pytorch_model.bin', '.gitattributes']


In [2]:
# Necessaries
import pandas as pd

data_dir = "/kaggle/input/competitions/rsna-knee-abnormality-detection"

train = pd.read_csv(f"{data_dir}/train.csv")
train_series = pd.read_csv(f"{data_dir}/train_series.csv")

finding_columns = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 
                    'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 
                    'Contusion', 'Fracture']

gold_data = train[train['Fracture'].notna()]
gold_labels = gold_data[['StudyInstanceUID'] + finding_columns].copy()

report_label = pd.read_csv("/kaggle/input/datasets/mayankptdr/report-labels/extracted_labels.csv")

all_labels = pd.concat([gold_labels, report_label], ignore_index=True)
all_labels[finding_columns] = all_labels[finding_columns].fillna(0)

print(all_labels.shape)

(4407, 13)


In [3]:
!pip install -q -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 82.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 78.7 MB/s eta 0:00:00:00:01


In [4]:
# Loading Transformer
from transformers import Dinov2Model, Dinov2Config

dinov2_path = '/kaggle/input/models/metaresearch/dinov2/pytorch/small/1'

dinov2 = Dinov2Model.from_pretrained(dinov2_path)
print(dinov2.config)
print(f"Hidden size: {dinov2.config.hidden_size}")

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Dinov2Config {
  "apply_layernorm": true,
  "architectures": [
    "Dinov2Model"
  ],
  "attention_probs_dropout_prob": 0.0,
  "drop_path_rate": 0.0,
  "dtype": "float32",
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.0,
  "hidden_size": 384,
  "image_size": 518,
  "initializer_range": 0.02,
  "layer_norm_eps": 1e-06,
  "layerscale_value": 1.0,
  "mlp_ratio": 4,
  "model_type": "dinov2",
  "num_attention_heads": 6,
  "num_channels": 3,
  "num_hidden_layers": 12,
  "out_features": [
    "stage12"
  ],
  "out_indices": [
    12
  ],
  "patch_size": 14,
  "qkv_bias": true,
  "reshape_hidden_states": true,
  "stage_names": [
    "stem",
    "stage1",
    "stage2",
    "stage3",
    "stage4",
    "stage5",
    "stage6",
    "stage7",
    "stage8",
    "stage9",
    "stage10",
    "stage11",
    "stage12"
  ],
  "transformers_version": "5.17.0",
  "use_mask_token": true,
  "use_swiglu_ffn": false
}

Hidden size: 384


 #### ****Data pipeline (mm based cropping + 6 slots) --- v1****

In [5]:
# Data pipeline (mm based cropping + 6 slots)
def select_series_6slots(study_id, train_series_df):
    study_series = train_series_df[train_series_df['StudyInstanceUID'] == study_id]
    slots = {}
    
    for plane in ['Sagittal', 'Coronal', 'Axial']:
        for fluid_sensitive in [1, 0]:
            key = f"{plane}_{fluid_sensitive}"
            candidates = study_series[
                (study_series['Anatomical_Plane'] == plane) & 
                (study_series['Fluid_Sensitive'] == fluid_sensitive)
            ]
            if len(candidates) > 0:
                slots[key] = candidates.iloc[0]['SeriesInstanceUID']
            else:
                slots[key] = None  # yeh-slot-is-study-mein-available-nahi
    
    return slots

# testing
test_slots = select_series_6slots(all_labels['StudyInstanceUID'].iloc[0], train_series)
print(test_slots)

{'Sagittal_1': '1.2.826.0.1.3680043.8.498.37372672584387115708078590109474719429', 'Sagittal_0': '1.2.826.0.1.3680043.8.498.10791858932231443712736629517371931282', 'Coronal_1': '1.2.826.0.1.3680043.8.498.21497837050858589852101556928971904198', 'Coronal_0': '1.2.826.0.1.3680043.8.498.12451507871565834868863189854063726441', 'Axial_1': '1.2.826.0.1.3680043.8.498.67495435033389756963800730529101229430', 'Axial_0': None}


#### ****Handling Missing Values****

In [6]:
# Missing counts
missing_counts = {f"{p}_{f}": 0 for p in ['Sagittal','Coronal','Axial'] for f in [1,0]}
total = 0

for sid in all_labels['StudyInstanceUID'].head(200):  # sample-pe-test, poore-4407-pe-baad-mein
    slots = select_series_6slots(sid, train_series)
    for key, val in slots.items():
        if val is None:
            missing_counts[key] += 1
    total += 1

print(f"Sample size: {total}")
for k, v in missing_counts.items():
    print(f"{k}: missing in {v}/{total} ({v/total*100:.1f}%)")

Sample size: 200
Sagittal_1: missing in 7/200 (3.5%)
Sagittal_0: missing in 5/200 (2.5%)
Coronal_1: missing in 8/200 (4.0%)
Coronal_0: missing in 43/200 (21.5%)
Axial_1: missing in 0/200 (0.0%)
Axial_0: missing in 167/200 (83.5%)


In [7]:
SLOTS = [('Sagittal', 1), ('Sagittal', 0), ('Coronal', 1), ('Coronal', 0), ('Axial', 1)]

def select_series_5slots(study_id, train_series_df):
    study_series = train_series_df[train_series_df['StudyInstanceUID'] == study_id]
    slots = {}
    
    for plane, fluid_sensitive in SLOTS:
        key = f"{plane}_{fluid_sensitive}"
        candidates = study_series[
            (study_series['Anatomical_Plane'] == plane) & 
            (study_series['Fluid_Sensitive'] == fluid_sensitive)
        ]
        slots[key] = candidates.iloc[0]['SeriesInstanceUID'] if len(candidates) > 0 else None
    
    return slots

test_slots_5 = select_series_5slots(all_labels['StudyInstanceUID'].iloc[0], train_series)
print(test_slots_5)

{'Sagittal_1': '1.2.826.0.1.3680043.8.498.37372672584387115708078590109474719429', 'Sagittal_0': '1.2.826.0.1.3680043.8.498.10791858932231443712736629517371931282', 'Coronal_1': '1.2.826.0.1.3680043.8.498.21497837050858589852101556928971904198', 'Coronal_0': '1.2.826.0.1.3680043.8.498.12451507871565834868863189854063726441', 'Axial_1': '1.2.826.0.1.3680043.8.498.67495435033389756963800730529101229430'}


 #### ****mm based cropping****

In [8]:
def crop_by_mm(pixel_array, pixel_spacing, crop_mm=130.0):
    
    row_spacing, col_spacing = pixel_spacing
    
    crop_h = int(crop_mm / row_spacing)
    crop_w = int(crop_mm / col_spacing)
    
    h, w = pixel_array.shape
    center_h, center_w = h // 2, w // 2
    
    half_h, half_w = crop_h // 2, crop_w // 2
    
    top = max(0, center_h - half_h)
    bottom = min(h, center_h + half_h)
    left = max(0, center_w - half_w)
    right = min(w, center_w + half_w)
    
    cropped = pixel_array[top:bottom, left:right]
    return cropped

In [9]:
#testing
import pydicom

sample_study = all_labels['StudyInstanceUID'].iloc[0]
sample_series = test_slots_5['Sagittal_1']
folder = f"{data_dir}/train_series/{sample_study}/{sample_series}"
sample_file = os.listdir(folder)[0]

dcm = pydicom.dcmread(f"{folder}/{sample_file}")
pixel_spacing = [float(x) for x in dcm.PixelSpacing]

print(f"Original shape: {dcm.pixel_array.shape}")
print(f"Pixel spacing: {pixel_spacing}")

cropped = crop_by_mm(dcm.pixel_array, pixel_spacing, crop_mm=130.0)
print(f"Cropped shape: {cropped.shape}")

Original shape: (320, 320)
Pixel spacing: [0.5, 0.5]
Cropped shape: (260, 260)


#### ****final-Resizing****

In [10]:
import cv2

def crop_and_resize(pixel_array, pixel_spacing, crop_mm=130.0, final_size=224):
    cropped = crop_by_mm(pixel_array, pixel_spacing, crop_mm)
    resized = cv2.resize(cropped, (final_size, final_size), interpolation=cv2.INTER_LINEAR)
    return resized

In [11]:
#testing
final = crop_and_resize(dcm.pixel_array, pixel_spacing, crop_mm=130.0, final_size=224)
print(f"Final shape: {final.shape}")

Final shape: (224, 224)


#### ****picking up multiple slices for DINOv2****

In [12]:
import numpy as np
def select_slice_indices(total_slices, n_slice=16, band=(0.12, 0.88)):
    start = int(total_slices * band[0])
    end = int(total_slices * band[1])
    indices = np.linspace(start, end-1, n_slice).astype(int)
    return indices

In [13]:
#testing
test_indices = select_slice_indices(total_slices=22, n_slice=16)
print(test_indices)

[ 2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 18]


### ****Dataset Class (5-slots, 16-slices/slot) v1****

In [14]:
import torch
from torch.utils.data import Dataset
import pydicom
import cv2
import numpy as np

class KneeMRIDatasetDINO(Dataset):
    def __init__(self, labels_df, train_series_df, data_dir, n_slice=16, crop_mm=130.0, final_size=224):
        self.labels_df = labels_df.reset_index(drop=True)
        self.train_series_df = train_series_df
        self.data_dir = data_dir
        self.n_slice = n_slice
        self.crop_mm = crop_mm
        self.final_size = final_size
        self.finding_columns = finding_columns
        self.slots = [('Sagittal', 1), ('Sagittal', 0), ('Coronal', 1), ('Coronal', 0), ('Axial', 1)]
    
    def __len__(self):
        return len(self.labels_df)
    
    def _load_slot(self, study_id, series_id):
        if series_id is None:
            # missing-slot: zero-tensor + mask=0
            return np.zeros((self.n_slice, self.final_size, self.final_size), dtype=np.float32), 0.0
        
        folder = f"{self.data_dir}/train_series/{study_id}/{series_id}"
        dcm_files = os.listdir(folder)
        slices = [pydicom.dcmread(f"{folder}/{f}") for f in dcm_files]
        slices_sorted = sorted(slices, key=lambda d: int(d.InstanceNumber))
        
        total = len(slices_sorted)
        indices = select_slice_indices(total, self.n_slice)
        
        pixel_spacing = [float(x) for x in slices_sorted[0].PixelSpacing]
        
        processed_slices = []
        for idx in indices:
            arr = slices_sorted[idx].pixel_array.astype(np.float32)
            arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-6)
            cropped_resized = crop_and_resize(arr, pixel_spacing, self.crop_mm, self.final_size)
            processed_slices.append(cropped_resized)
        
        volume = np.stack(processed_slices).astype(np.float32)
        return volume, 1.0  # mask=1, slot-available
    
    def __getitem__(self, idx):
        row = self.labels_df.iloc[idx]
        study_id = row['StudyInstanceUID']
        
        slot_map = select_series_5slots(study_id, self.train_series_df)
        
        slot_volumes = []
        slot_masks = []
        for plane, fs in self.slots:
            key = f"{plane}_{fs}"
            series_id = slot_map[key]
            vol, mask = self._load_slot(study_id, series_id)
            slot_volumes.append(vol)
            slot_masks.append(mask)
        
        slots_tensor = torch.tensor(np.stack(slot_volumes), dtype=torch.float32) 
        mask_tensor = torch.tensor(slot_masks, dtype=torch.float32)  
        
        labels = torch.tensor(row[self.finding_columns].values.astype(float), dtype=torch.float32)
        
        return {'slots': slots_tensor, 'mask': mask_tensor, 'labels': labels, 'study_id': study_id}

In [15]:
#testing
dataset_dino = KneeMRIDatasetDINO(all_labels, train_series, data_dir)
sample = dataset_dino[0]
print(sample['slots'].shape)
print(sample['mask'])
print(sample['labels'])

torch.Size([5, 16, 224, 224])
tensor([1., 1., 1., 1., 1.])
tensor([0., 0., 0., 0., 0., 0., 1., 1., 0., 0., 0., 0.])


#### ****Catched data****

In [16]:
print(os.listdir('/kaggle/input/datasets/mayankptdr'))

['d-rsna-4', 'report-labels', 'd-rsna-3', 'd-rsna-5', 'd-rsna-2', 'd-rsna-1']


In [17]:
base = '/kaggle/input/datasets/mayankptdr'

cache_dirs_dino = []
total_files = 0

for name in ['d-rsna-1', 'd-rsna-2', 'd-rsna-3', 'd-rsna-4', 'd-rsna-5']:
    path = f"{base}/{name}/cached_dino"
    count = len(os.listdir(path))
    print(f"{name}: {count} files")
    cache_dirs_dino.append(path)
    total_files += count

print(f"\nTotal cached studies: {total_files}")

d-rsna-1: 881 files
d-rsna-2: 881 files
d-rsna-3: 881 files
d-rsna-4: 881 files
d-rsna-5: 883 files

Total cached studies: 4407


### ****Dataset Class v2****

In [18]:
import torch
from torch.utils.data import Dataset
import numpy as np

class KneeMRIDatasetDINOCached(Dataset):
    def __init__(self, labels_df, cache_dirs):
        self.labels_df = labels_df.reset_index(drop=True)
        self.cache_dirs = cache_dirs
        self.finding_columns = finding_columns
        
        self.valid_indices = []
        for idx, row in self.labels_df.iterrows():
            if self._find_cache_path(row['StudyInstanceUID']) is not None:
                self.valid_indices.append(idx)
        
        print(f"Valid studies (cache found): {len(self.valid_indices)} / {len(self.labels_df)}")
    
    def _find_cache_path(self, study_id):
        for d in self.cache_dirs:
            path = f"{d}/{study_id}.npz"
            if os.path.exists(path):
                return path
        return None
    
    def __len__(self):
        return len(self.valid_indices)
    
    def __getitem__(self, i):
        idx = self.valid_indices[i]
        row = self.labels_df.iloc[idx]
        study_id = row['StudyInstanceUID']
        
        cache_path = self._find_cache_path(study_id)
        data = np.load(cache_path)
        
        slots = torch.tensor(data['slots'].astype(np.float32), dtype=torch.float32) 
        mask = torch.tensor(data['mask'], dtype=torch.float32)
        labels = torch.tensor(row[self.finding_columns].values.astype(float), dtype=torch.float32)
        
        return {'slots': slots, 'mask': mask, 'labels': labels, 'study_id': study_id}

dataset_dino = KneeMRIDatasetDINOCached(all_labels, cache_dirs_dino)
sample = dataset_dino[0]
print(sample['slots'].shape, sample['mask'], sample['labels'])

Valid studies (cache found): 4407 / 4407
torch.Size([5, 16, 224, 224]) tensor([1., 1., 1., 1., 1.]) tensor([0., 0., 0., 0., 0., 0., 1., 1., 0., 0., 0., 0.])


#### ****DINOv2 (training test layer1)****

In [19]:
import torch
import torch.nn as nn
from transformers import Dinov2Model

class SliceAttentionPool(nn.Module):
    def __init__(self, dim=384):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(dim, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )
    
    def forward(self, x):
        weights = self.attn(x) 
        weights = torch.softmax(weights, dim=1)
        pooled = (x * weights).sum(dim=1)  
        return pooled

class DINOv2KneeModel(nn.Module):
    def __init__(self, dinov2_path, num_findings=12, n_slots=5, n_slice=16):
        super().__init__()
        self.dinov2 = Dinov2Model.from_pretrained(dinov2_path)
        self.dim = self.dinov2.config.hidden_size  # 384
        self.n_slots = n_slots
        self.n_slice = n_slice
        
        self.slice_pool = SliceAttentionPool(self.dim)
        
        self.slot_attn = nn.Sequential(
            nn.Linear(self.dim, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )
        
        self.head = nn.Sequential(
            nn.Linear(self.dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_findings)
        )
    
    def forward(self, slots, mask):
       
        batch_size = slots.shape[0]
        
       
        x = slots.view(batch_size * self.n_slots * self.n_slice, 1, 224, 224)
        x = x.repeat(1, 3, 1, 1)  
        
        features = self.dinov2(pixel_values=x).pooler_output 
        features = features.view(batch_size, self.n_slots, self.n_slice, self.dim)
        
        # attention pooling (slice-level)
        slot_features = []
        for s in range(self.n_slots):
            pooled = self.slice_pool(features[:, s, :, :])  
            slot_features.append(pooled)
        slot_features = torch.stack(slot_features, dim=1)  
        
        # attention, masked (slot-level)
        slot_weights = self.slot_attn(slot_features).squeeze(-1)  
        slot_weights = slot_weights.masked_fill(mask == 0, float('-inf'))
        slot_weights = torch.softmax(slot_weights, dim=1).unsqueeze(-1) 
        
        final_feature = (slot_features * slot_weights).sum(dim=1)  
        
        return self.head(final_feature)

In [20]:
#testing
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DINOv2KneeModel(dinov2_path).to(device)

from torch.utils.data import DataLoader
loader = DataLoader(dataset_dino, batch_size=1, shuffle=True)
batch = next(iter(loader))

slots = batch['slots'].to(device)
mask = batch['mask'].to(device)

output = model(slots, mask)
print(output.shape)

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

torch.Size([1, 12])


#### ****DINOv2 Training loop layer****

In [21]:
# Model + Pooling classes
import torch
import torch.nn as nn
from transformers import Dinov2Model

class SliceAttentionPool(nn.Module):
    def __init__(self, dim=384):
        super().__init__()
        self.attn = nn.Sequential(nn.Linear(dim, 128), nn.Tanh(), nn.Linear(128, 1))
    def forward(self, x):
        weights = torch.softmax(self.attn(x), dim=1)
        return (x * weights).sum(dim=1)

class DINOv2KneeModel(nn.Module):
    def __init__(self, dinov2_path, num_findings=12, n_slots=5, n_slice=16):
        super().__init__()
        self.dinov2 = Dinov2Model.from_pretrained(dinov2_path)
        self.dim = self.dinov2.config.hidden_size
        self.n_slots = n_slots
        self.n_slice = n_slice
        self.slice_pool = SliceAttentionPool(self.dim)
        self.slot_attn = nn.Sequential(nn.Linear(self.dim, 128), nn.Tanh(), nn.Linear(128, 1))
        self.head = nn.Sequential(nn.Linear(self.dim, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, num_findings))
    
    def forward(self, slots, mask):
        batch_size = slots.shape[0]
        x = slots.view(batch_size * self.n_slots * self.n_slice, 1, 224, 224).repeat(1, 3, 1, 1)
        features = self.dinov2(pixel_values=x).pooler_output
        features = features.view(batch_size, self.n_slots, self.n_slice, self.dim)
        
        slot_features = torch.stack([self.slice_pool(features[:, s, :, :]) for s in range(self.n_slots)], dim=1)
        
        slot_weights = self.slot_attn(slot_features).squeeze(-1)
        slot_weights = slot_weights.masked_fill(mask == 0, float('-inf'))
        slot_weights = torch.softmax(slot_weights, dim=1).unsqueeze(-1)
        
        final_feature = (slot_features * slot_weights).sum(dim=1)
        return self.head(final_feature)

In [22]:
!pip install -q iterative-stratification

In [23]:
# model setup (folds, etc)

from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

valid_labels = all_labels.iloc[dataset_dino.valid_indices].reset_index(drop=True)
X = valid_labels['StudyInstanceUID'].values
y = valid_labels[finding_columns].values

mskf = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds = []
for fold_num, (train_idx, val_idx) in enumerate(mskf.split(X, y)):
    folds.append({'train_idx': train_idx, 'val_idx': val_idx})

pos_counts = valid_labels[finding_columns].sum().values
neg_counts = len(valid_labels) - pos_counts
pos_weights = torch.tensor(neg_counts / (pos_counts + 1e-6), dtype=torch.float32)
print("Setup done")

Setup done


In [24]:
import shutil, os

old_path = '/kaggle/input/notebooks/mayankptdr/dinov2-rsna/checkpoints_dino'  # exact-naam-verify-karo-upar-se
checkpoint_dir_dino = "/kaggle/working/checkpoints_dino"
os.makedirs(checkpoint_dir_dino, exist_ok=True)

for f in os.listdir(old_path):
    shutil.copy(f"{old_path}/{f}", f"{checkpoint_dir_dino}/{f}")

print("Checkpoints restored:", os.listdir(checkpoint_dir_dino))

# fold_results.json bhi restore karo
old_results = '/kaggle/input/notebooks/mayankptdr/dinov2-rsna/fold_results_dino.json'
shutil.copy(old_results, "/kaggle/working/fold_results_dino.json")
print("Results restored")

Checkpoints restored: ['fold3_best.pt', 'fold3_latest.pt', 'fold4_latest.pt', 'fold4_best.pt', 'fold0_best.pt', 'fold2_best.pt', 'fold2_latest.pt', 'fold0_latest.pt', 'fold1_best.pt', 'fold1_latest.pt']
Results restored


In [25]:
# Training Function

import torch
import torch.nn as nn
from torch.amp import autocast, GradScaler
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import roc_auc_score
import numpy as np
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
checkpoint_dir_dino = "/kaggle/working/checkpoints_dino"
os.makedirs(checkpoint_dir_dino, exist_ok=True)

def train_one_fold_dino(fold_num, train_idx, val_idx, epochs=8, batch_size=1, lr=1e-4, accumulation_steps=4):
    train_subset = Subset(dataset_dino, train_idx)
    val_subset = Subset(dataset_dino, val_idx)
    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=2)
    
    model = DINOv2KneeModel(dinov2_path, num_findings=12).to(device)
    
    for param in model.dinov2.parameters():
        param.requires_grad = False
    for param in model.dinov2.encoder.layer[-2:].parameters():
        param.requires_grad = True
    
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights.to(device))
    # === AMP/GradScaler HATA DIYA — FULL FP32 TRAINING ===
    
    checkpoint_path = f"{checkpoint_dir_dino}/fold{fold_num}_latest.pt"
    best_path = f"{checkpoint_dir_dino}/fold{fold_num}_best.pt"
    start_epoch, best_auroc = 0, 0
    
    if os.path.exists(checkpoint_path):
        ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
        model.load_state_dict(ckpt['model_state'])
        optimizer.load_state_dict(ckpt['optimizer_state'])
        start_epoch = ckpt['epoch'] + 1
        best_auroc = ckpt['best_auroc']
        print(f"Resumed fold {fold_num} from epoch {start_epoch}")
    
    for epoch in range(start_epoch, epochs):
        model.train()
        train_loss = 0
        nan_skips = 0
        optimizer.zero_grad()
        for step, batch in enumerate(train_loader):
            slots = batch['slots'].to(device)
            mask = batch['mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(slots, mask)
            loss = criterion(outputs, labels) / accumulation_steps
            
            if torch.isnan(loss) or torch.isinf(loss):
                nan_skips += 1
                optimizer.zero_grad()
                continue
            
            loss.backward()
            
            if (step + 1) % accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()
            
            train_loss += loss.item() * accumulation_steps
        
        if nan_skips > 0:
            print(f"  (skipped {nan_skips} NaN/Inf batches this epoch)")
        
        model.eval()
        all_preds, all_labels_val = [], []
        with torch.no_grad():
            for batch in val_loader:
                slots = batch['slots'].to(device)
                mask = batch['mask'].to(device)
                labels = batch['labels'].to(device)
                
                outputs = torch.sigmoid(model(slots, mask))
                all_preds.append(outputs.cpu().numpy())
                all_labels_val.append(labels.cpu().numpy())
        
        all_preds = np.concatenate(all_preds)
        all_labels_val = np.concatenate(all_labels_val)
        aurocs = []
        for i in range(12):
            try:
                aurocs.append(roc_auc_score(all_labels_val[:, i], all_preds[:, i]))
            except ValueError:
                aurocs.append(0.5)
        
        macro_auroc = np.mean(aurocs)
        avg_loss = train_loss / len(train_loader)
        print(f"Fold {fold_num} | Epoch {epoch+1}/{epochs} | Train Loss: {avg_loss:.4f} | Val AUROC: {macro_auroc:.4f}")
        
        torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                    'optimizer_state': optimizer.state_dict(), 'best_auroc': max(best_auroc, macro_auroc)},
                   checkpoint_path)
        
        if macro_auroc > best_auroc:
            best_auroc = macro_auroc
            torch.save({'model_state': model.state_dict(), 'epoch': epoch,
                        'auroc': macro_auroc, 'per_finding_auroc': dict(zip(finding_columns, aurocs))}, best_path)
            print(f"  --> New best! Saved to {best_path}")
    
    return best_auroc

In [26]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import roc_auc_score
import numpy as np
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
checkpoint_dir_dino = "/kaggle/working/checkpoints_dino"
os.makedirs(checkpoint_dir_dino, exist_ok=True)

def train_one_fold_dino(fold_num, train_idx, val_idx, epochs=8, batch_size=1, lr=1e-4, accumulation_steps=4):
    train_subset = Subset(dataset_dino, train_idx)
    val_subset = Subset(dataset_dino, val_idx)
    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=2)
    
    model = DINOv2KneeModel(dinov2_path, num_findings=12).to(device)
    
    for param in model.dinov2.parameters():
        param.requires_grad = False
    for param in model.dinov2.encoder.layer[-2:].parameters():
        param.requires_grad = True
    
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights.to(device))
    
    checkpoint_path = f"{checkpoint_dir_dino}/fold{fold_num}_latest.pt"
    best_path = f"{checkpoint_dir_dino}/fold{fold_num}_best.pt"
    start_epoch, best_auroc = 0, 0
    
    if os.path.exists(checkpoint_path):
        ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
        model.load_state_dict(ckpt['model_state'])
        optimizer.load_state_dict(ckpt['optimizer_state'])
        start_epoch = ckpt['epoch'] + 1
        best_auroc = ckpt['best_auroc']
        print(f"Resumed fold {fold_num} from epoch {start_epoch}")
    
    for epoch in range(start_epoch, epochs):
        model.train()
        train_loss = 0
        nan_skips = 0
        optimizer.zero_grad()
        for step, batch in enumerate(train_loader):
            slots = batch['slots'].to(device)
            mask = batch['mask'].to(device)
            labels = batch['labels'].to(device)
            
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                outputs = model(slots, mask)
                loss = criterion(outputs, labels) / accumulation_steps
            
            if torch.isnan(loss) or torch.isinf(loss):
                nan_skips += 1
                optimizer.zero_grad()
                continue
            
            loss.backward()
            
            if (step + 1) % accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()
            
            train_loss += loss.item() * accumulation_steps
            
            if (step + 1) % 200 == 0:
                print(f"    step {step+1}/{len(train_loader)}, running_loss={train_loss/(step+1):.4f}")
        
        if nan_skips > 0:
            print(f"  (skipped {nan_skips} NaN/Inf batches this epoch)")
        
        model.eval()
        all_preds, all_labels_val = [], []
        with torch.no_grad():
            for batch in val_loader:
                slots = batch['slots'].to(device)
                mask = batch['mask'].to(device)
                labels = batch['labels'].to(device)
                
                with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                    outputs = torch.sigmoid(model(slots, mask))
                all_preds.append(outputs.float().cpu().numpy())
                all_labels_val.append(labels.cpu().numpy())
        
        all_preds = np.concatenate(all_preds)
        all_labels_val = np.concatenate(all_labels_val)
        aurocs = []
        for i in range(12):
            try:
                aurocs.append(roc_auc_score(all_labels_val[:, i], all_preds[:, i]))
            except ValueError:
                aurocs.append(0.5)
        
        macro_auroc = np.mean(aurocs)
        avg_loss = train_loss / len(train_loader)
        print(f"Fold {fold_num} | Epoch {epoch+1}/{epochs} | Train Loss: {avg_loss:.4f} | Val AUROC: {macro_auroc:.4f}")
        
        torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                    'optimizer_state': optimizer.state_dict(), 'best_auroc': max(best_auroc, macro_auroc)},
                   checkpoint_path)
        
        if macro_auroc > best_auroc:
            best_auroc = macro_auroc
            torch.save({'model_state': model.state_dict(), 'epoch': epoch,
                        'auroc': macro_auroc, 'per_finding_auroc': dict(zip(finding_columns, aurocs))}, best_path)
            print(f"  --> New best! Saved to {best_path}")
    
    del model
    torch.cuda.empty_cache()
    
    return best_auroc

In [27]:
# Resumable loop (8-epochs, 5 folds)

import json

fold_results_path = "/kaggle/working/fold_results_dino.json"

if os.path.exists(fold_results_path):
    with open(fold_results_path) as f:
        fold_results = json.load(f)
    fold_results = {int(k): v for k, v in fold_results.items()}
else:
    fold_results = {}

print(f"Already-completed-folds: {list(fold_results.keys())}")

for fold_num in range(5):
    if fold_num in fold_results:
        print(f"Fold {fold_num} already completed (AUROC={fold_results[fold_num]:.4f}), skipping")
        continue
    
    print(f"\n{'='*50}\nStarting Fold {fold_num}\n{'='*50}")
    
    best_auroc = train_one_fold_dino(
        fold_num=fold_num,
        train_idx=folds[fold_num]['train_idx'],
        val_idx=folds[fold_num]['val_idx'],
        epochs=8,
        batch_size=1
    )
    
    fold_results[fold_num] = best_auroc
    with open(fold_results_path, 'w') as f:
        json.dump(fold_results, f)
    
    print(f"Fold {fold_num} complete. Best AUROC: {best_auroc:.4f}")

print(f"\nAll done: {fold_results}")
if fold_results:
    print(f"Average AUROC: {np.mean(list(fold_results.values())):.4f}")

Already-completed-folds: [0, 1, 2, 3]
Fold 0 already completed (AUROC=0.7756), skipping
Fold 1 already completed (AUROC=0.7955), skipping
Fold 2 already completed (AUROC=0.7930), skipping
Fold 3 already completed (AUROC=0.7895), skipping

Starting Fold 4


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Resumed fold 4 from epoch 2
    step 200/3525, running_loss=0.9719
    step 400/3525, running_loss=0.9383
    step 600/3525, running_loss=0.9236
    step 800/3525, running_loss=0.9148
    step 1000/3525, running_loss=0.9132
    step 1200/3525, running_loss=0.9113
    step 1400/3525, running_loss=0.9112
    step 1600/3525, running_loss=0.9095
    step 1800/3525, running_loss=0.9049
    step 2000/3525, running_loss=0.9023
    step 2200/3525, running_loss=0.8990
    step 2400/3525, running_loss=0.8997
    step 2600/3525, running_loss=0.9029
    step 2800/3525, running_loss=0.9003
    step 3000/3525, running_loss=0.8992
    step 3200/3525, running_loss=0.8995
    step 3400/3525, running_loss=0.8991
Fold 4 | Epoch 3/8 | Train Loss: 0.8979 | Val AUROC: 0.7480
  --> New best! Saved to /kaggle/working/checkpoints_dino/fold4_best.pt
    step 200/3525, running_loss=0.8501
    step 400/3525, running_loss=0.8840
    step 600/3525, running_loss=0.8835
    step 800/3525, running_loss=0.8711
    step

In [28]:
print("hello")

hello
